In [1]:
import pandas as pd
import tensorflow as tf
import numpy as np
import cv2
import os
import matplotlib
import matplotlib.pyplot as plt
import random

print("Versión de pandas:", pd.__version__)
print("Versión de tensorflow:", tf.__version__)
print("Versión de numpy:", np.__version__)
print("Versión de OpenCV:", cv2.__version__)
print("Versión de matplotlib:", matplotlib.__version__)
print("Sistema operativo:", os.name)


Versión de pandas: 2.0.3
Versión de tensorflow: 2.7.0
Versión de numpy: 1.24.4
Versión de OpenCV: 4.11.0
Versión de matplotlib: 3.7.5
Sistema operativo: nt


In [19]:
def gpu_setup():
    """
    Configures TensorFlow to use available GPUs with memory growth enabled.

    This function checks for available GPUs on the system and enables memory growth
    for each GPU.If no GPUs are found, a message is printed indicating their absence. 

    Returns:
        None
    """
    gpus = tf.config.experimental.list_physical_devices('GPU')
    if gpus: 
        try:
            for gpu in gpus:
                tf.config.experimental.set_memory_growth(gpu, True)
        except RuntimeError as e:
            print("Error configuring GPU memory growth:", {e})
    else:
        print("No GPUs found.")

In [20]:
def cargar_imagenes_y_masks(fnames, ruta, normalizar=False, es_mask=False):

    datos = []
    for fname in fnames:
        img = cv2.imread(ruta + "\\" + fname, 0)
        img = cv2.resize(img,(320,320))
        if es_mask:
            img = np.where(img >= 5, 4, img)
        img = np.expand_dims(img, axis=-1)
        if normalizar:
            img = img / 255.0
        datos.append(img)
    dtype = np.float32 if normalizar else np.uint8
    return np.array(datos, dtype=dtype)

In [21]:
def data_sort(path_Imgs,path_Msks,path_csv):
    """
    Sorts the input data based on the specified column.

    Args:
        data (pd.DataFrame): The input data to be sorted.

    Returns:
        pd.DataFrame: The sorted data.
    """

    random_state_value = random.randint(1, 250)

    df = pd.read_csv(path_csv)
    df = df.sample(frac=1, random_state=random_state_value)

    x_train_fnames = df['imgs'].values[0:int(0.7*df.shape[0])]
    y_train_fnames = df['msks'].values[0:int(0.7*df.shape[0])]

    x_val_fnames = df['imgs'].values[int(0.7*df.shape[0]) :int(0.7*df.shape[0]) +int(0.15*df.shape[0]) ]
    y_val_fnames = df['msks'].values[int(0.7*df.shape[0]) :int(0.7*df.shape[0]) +int(0.15*df.shape[0]) ]

    x_test_fnames = df['imgs'].values[int(0.7*df.shape[0]) +int(0.15*df.shape[0]) :]
    y_test_fnames = df['msks'].values[int(0.7*df.shape[0]) +int(0.15*df.shape[0]) :]

    x_train = cargar_imagenes_y_masks(x_train_fnames, path_Imgs, normalizar=True)
    x_test = cargar_imagenes_y_masks(x_test_fnames, path_Imgs, normalizar=True)
    x_val = cargar_imagenes_y_masks(x_val_fnames, path_Imgs, normalizar=True)

    y_train = cargar_imagenes_y_masks(y_train_fnames, path_Msks, es_mask=True)
    y_test = cargar_imagenes_y_masks(y_test_fnames, path_Msks, es_mask=True)
    y_val = cargar_imagenes_y_masks(y_val_fnames, path_Msks, es_mask=True)

    return x_train, y_train, x_val, y_val, x_test, y_test

In [22]:
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Conv2DTranspose, concatenate, Dropout
from tensorflow.keras.models import Model

def create_unet_model():
    """
    Creates a multimodal neural network model for regression tasks.

    Args:
        
    Returns:
        keras.Model: Compiled multimodal model.
    """
    # ===============
    # Entrada
    Image_input = Input(shape=(320, 320, 1)) 

    # ===============
    # Codificador

    # conv1
    conv1 = Conv2D(128, (5,5), activation='relu', padding='same')(Image_input)
    conv1 = Dropout(0.2)(conv1) #opcional
    conv1 = Conv2D(128, (5,5), activation='relu', padding='same')(conv1)
    maxp1 = MaxPooling2D((2, 2))(conv1)

    # conv2
    conv2 = Conv2D(64, (5,5), activation='relu', padding='same')(maxp1)
    conv2 = Dropout(0.2)(conv2) #opcional
    conv2 = Conv2D(64, (5,5), activation='relu', padding='same')(conv2)
    maxp2 = MaxPooling2D((2, 2))(conv2)

    # conv3
    conv3 = Conv2D(32, (5,5), activation='relu', padding='same')(maxp2)
    conv3 = Dropout(0.2)(conv3) #opcional
    conv3 = Conv2D(32, (5,5), activation='relu', padding='same')(conv3)
    maxp3 = MaxPooling2D((2, 2))(conv3)

    # conv4
    conv4 = Conv2D(16, (5,5), activation='relu', padding='same')(maxp3)
    conv4 = Dropout(0.2)(conv4) #opcional
    conv4 = Conv2D(16, (5,5), activation='relu', padding='same')(conv4)
    maxp4 = MaxPooling2D(pool_size=(2, 2))(conv4)

    # conv5
    conv5 = Conv2D(8, (5,5), activation='relu', padding='same')(maxp4)
    conv5 = Dropout(0.3)(conv5) #opcional
    conv5 = Conv2D(8, (5,5), activation='relu', padding='same')(conv5)


    # ===============
    # Decodificador

    # dec1
    dec1 = Conv2DTranspose(16, (2, 2), strides=(2, 2), padding='same')(conv5)
    dec1 = concatenate([dec1, conv4])
    dec1 = Conv2D(16, (3,3), activation='relu', padding='same')(dec1)
    dec1 = Dropout(0.2)(dec1) #opcional
    dec1 = Conv2D(16, (3,3), activation='relu', padding='same')(dec1)


    # dec2
    dec2 = Conv2DTranspose(32, (2, 2), strides=(2, 2), padding='same')(dec1)
    dec2 = concatenate([dec2, conv3])
    dec2 = Conv2D(32, (3,3), activation='relu', padding='same')(dec2)
    dec2 = Dropout(0.2)(dec2) #opcional
    dec2 = Conv2D(32, (3,3), activation='relu', padding='same')(dec2)

    # dec3
    dec3 = Conv2DTranspose(64, (2, 2), strides=(2, 2), padding='same')(dec2)
    dec3 = concatenate([dec3, conv2])
    dec3 = Conv2D(64, (3,3), activation='relu', padding='same')(dec3)
    dec3 = Dropout(0.2)(dec3) #opcional
    dec3 = Conv2D(64, (3,3), activation='relu', padding='same')(dec3)

    #dec 4
    dec4 = Conv2DTranspose(128, (2, 2), strides=(2, 2), padding='same')(dec3)
    dec4 = concatenate([dec4, conv1], axis=3)
    dec4 = Conv2D(128, (3,3), activation='relu', padding='same')(dec4)
    dec4 = Dropout(0.2)(dec4) #opcional
    dec4 = Conv2D(128, (3,3), activation='relu', padding='same')(dec4)


    # ===============
    # Salida

    outputs = Conv2D(5, (1, 1), activation='softmax')(dec4)


    # ===============
    # Interconectar todo en un modelo

    unet = tf.keras.models.Model(inputs=Image_input, outputs=outputs)

    unet.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

    return unet
    

In [23]:
def train_model(unet,x_train,y_train,x_test,y_test):
    """
    Trains the multimodal model.

    Args:
        model (keras.Model): The multimodal model to train.
        x_train_xray (numpy.ndarray): Training X-ray images.
        x_train_msks (numpy.ndarray): Training masks.
        x_train_wavelet (numpy.ndarra
    """

    from tensorflow.keras.callbacks import EarlyStopping
    early_stop = EarlyStopping(monitor="val_loss", patience=30)

    with tf.device('/CPU:0'):
        dataset_train = tf.data.Dataset.from_tensor_slices((x_train,y_train)).batch(4).prefetch(tf.data.AUTOTUNE)
        dataset_test = tf.data.Dataset.from_tensor_slices((x_test,y_test)).batch(4).prefetch(tf.data.AUTOTUNE) 

    unet.fit(dataset_train, epochs=1000, validation_data=dataset_test,callbacks=[early_stop])

    # unet.save(r"D:\Trabajo_Grado\Algoritmos\Red_multimodal_MNN\Metricas\7_hidden_layers_no_skip_connection\Img_Msks_wavelet\mmn_model.h5")
    return unet

In [24]:
import matplotlib.pyplot as plt

def plot_training_metrics(losses):
    """
    Plots training metrics (MAPE and Loss) in a single figure with two subplots.

    Args:
        losses (pd.DataFrame): DataFrame containing training and validation metrics.
                               Expected columns: ["mape", "val_mape", "loss", "val_loss"].
    """
    fig, axes = plt.subplots(1, 2, figsize=(12, 5)) 

    axes[0].plot(losses["mape"], label="Train MAPE")
    axes[0].plot(losses["val_mape"], label="Validation MAPE")
    axes[0].set_title("Error Porcentual Absoluto Medio")
    axes[0].set_xlabel("Épocas")
    axes[0].set_ylabel("MAPE (Error Porcentual Absoluto Medio)")
    axes[0].legend()

    axes[1].plot(losses["loss"], label="Train Loss")
    axes[1].plot(losses["val_loss"], label="Validation Loss")
    axes[1].set_title("Evolución de la Pérdida durante el Entrenamiento")
    axes[1].set_xlabel("Épocas")
    axes[1].set_ylabel("Pérdida")
    axes[1].legend()

    plt.tight_layout() 
    # plt.savefig(r"D:\Trabajo_Grado\Algoritmos\Red_multimodal_MNN\Metricas\7_hidden_layers_no_skip_connection\Img_Msks_wavelet\mape_loss_plot.png", dpi=300, bbox_inches="tight")
    plt.show()

In [25]:
from sklearn.metrics import f1_score

def dice_sklearn(y_true, y_pred, num_classes):
    dice_scores = []
    for i in range(num_classes):
        y_true_class = (y_true == i).astype(int)
        y_pred_class = (y_pred == i).astype(int)
        dice = f1_score(y_true_class.ravel(), y_pred_class.ravel())  # Asegurar 1D
        dice_scores.append(dice)
    return dice_scores

In [26]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

def evaluate_model(unet,x_val, y_val):
    """
    Evaluates the trained model on the test dataset.

    Args:
        model (keras.Model): The trained multimodal model.
        x_test_xray (numpy.ndarray): Test X-ray images.
        x_test_msks (numpy.ndarray): Test masks.
        x_test_wavelet (numpy.ndarray)
    """

    print(unet.metrics_names)
    losses = pd.DataFrame(unet.history.history)
    # losses.to_csv(r"D:\Trabajo_Grado\Algoritmos\Red_multimodal_MNN\Metricas\7_hidden_layers_no_skip_connection\Img_Msks_wavelet\losses.csv", index=False)
    plot_training_metrics(losses)

    with tf.device('/CPU:0'): 
        preds = unet.predict(x_val)    
    
    y_preds = np.argmax(preds, axis=-1)
    dice_scores = dice_sklearn(y_val, y_preds, num_classes=5)  # Asumiendo que ya tienes estas variables
    dice_global = np.mean(dice_scores)

    dice_scores = [round(score, 4) for score in dice_scores]
    dice_global = round(dice_global, 4)

    y_val_flat = y_val.flatten()
    y_preds_flat = y_preds.flatten()

    cm = confusion_matrix(y_val_flat, y_preds_flat, labels=np.arange(5))    

    ConfusionMatrixDisplay(cm, display_labels=[f"Clase {i}" for i in range(5)]).plot(cmap="Blues", colorbar=True, values_format='d')
    plt.title("Matriz de confusión por clase")
    # plt.savefig(r"D:\Trabajo_Grado\Algoritmos\UNet\no_cross_validation\Metricas_prototipo_2_UNet\UNet_9\Matriz confusion.png")
    plt.show()

    return dice_scores,dice_global,cm

In [27]:
def unet():
    path_Imgs = r'D:\Trabajo_Grado\Algoritmos\Datasets\pierna_derecha\dataset_augmented\Imgs'
    path_Msks = r'D:\Trabajo_Grado\Algoritmos\Datasets\pierna_derecha\dataset_augmented\Msks'
    path_csv = r'D:\Trabajo_Grado\Algoritmos\Datasets\pierna_derecha\dataset_augmented\dataset_augmented.csv'

    gpu_setup()
    x_train, y_train, x_val, y_val, x_test, y_test = data_sort(path_Imgs,path_Msks,path_csv)
    unet = create_unet_model()
    unet.summary()
    unet = train_model(unet,x_train,y_train,x_test,y_test)
    dice_scores,dice_global,cm = evaluate_model (unet,x_val, y_val)
    
    print("Dice por clases:", dice_scores)
    print("Dice global:", dice_global)
    print("Cantidad de pixeles clasificados correctamente es: ",np.trace(cm))

In [29]:
unet()

KeyboardInterrupt: 